# 02 · Preprocessing and cleaning

**Why.** Make the features reliable without throwing away real respondents. The rules: stop on anything missing or invalid instead of guessing, keep every original code, and report odd answers rather than delete them.

**Re-run it:** the same `diabetes-screening run` command as notebook 01; this stage writes `cleaning.json` and the row-level `cleaned.csv` (kept out of git).

In [1]:
# Every notebook reads the committed aggregate reports, so it runs from a fresh clone without the survey data.
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENDPOINT = "combined"  # "prediabetes" shows the sensitivity analysis instead
REPORTS = ROOT / "reports"
STAGE_1_4 = REPORTS / ("uploaded-combined" if ENDPOINT == "combined" else "prediabetes")
MODELING = REPORTS / "modeling" / ENDPOINT
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def figure(path, width=860):
    display(Image(filename=str(path), width=width))

In [2]:
cleaning = read_json(STAGE_1_4 / "cleaning.json")
pd.Series({key: cleaning[key] for key in ["input_rows", "output_rows", "duplicate_policy", "duplicate_rows",
                                          "missingness_policy", "invalid_values_policy", "duplicate_rationale",
                                          "transformations"]}).to_frame("decision")

,decision
input_rows,253680
output_rows,253680
duplicate_policy,keep
duplicate_rows,23968
missingness_policy,Fail on missing values; never silently drop or impute respondents
invalid_values_policy,"Fail on noninteger, nonfinite, out-of-codebook values"
duplicate_rationale,Identical profiles may be different respondents; no respondent IDs are available
transformations,"Preserve original codes and BMI/health days; no clipping, scaling, or balancing"


## Answers worth a second look

These are valid codes that look unusual. They are counted, not changed: the survey's BMI range is the observed source range, and a 30 in the health-day questions can mean "every day".

In [3]:
pd.Series(cleaning["review_flags"]).to_frame("respondents")

,respondents
HighChol=1 but CholCheck=0 (told cholesterol high without a check in five years),1981
BMI below 15,68
BMI 50 or above,2547
BMI 60 or above,868
MentHlth == 30 (heaping at the top code),12088
PhysHlth == 30 (heaping at the top code),19400


## BMI and health days

In [4]:
pd.DataFrame(cleaning["distribution_summary"]).T

,count,mean,std,min,1%,25%,50%,75%,99%,max
BMI,253680.0,28.382,6.609,12.0,18.0,24.0,27.0,31.0,50.0,98.0
MentHlth,253680.0,3.185,7.413,0.0,0.0,0.0,0.0,2.0,30.0,30.0
PhysHlth,253680.0,4.242,8.718,0.0,0.0,0.0,0.0,3.0,30.0,30.0


**Takeaway.** Nothing had to be imputed or dropped. Identical answer profiles are kept, because the file has no respondent ID that could show they are accidental copies; stage 5 tests whether they distort model scores (they don't). No scaling or balancing happens here: those belong inside the model pipelines in stage 5, where they can't leak information from the test set.

More detail: [cleaning decisions](../docs/cleaning.md). This is a research prototype for risk stratification, not a diagnostic or clinical decision tool. The outcome is self-reported, the data are cross-sectional and unweighted, and associations are not causes.